In [3]:
import pandas as pd
import os

# 1. 读取原始 AIBL.csv 文件
input_file = 'AIBL.csv'

# 检查文件是否存在
if not os.path.exists(input_file):
    print(f"错误: 找不到文件 {input_file}，请确保它在当前目录下。")
else:
    print("正在读取数据...")
    df = pd.read_csv(input_file)

    # 2. 根据标签列进行筛选切分
    # 逻辑：如果某行的 'NC' 列为 1，则归为 NC 类，以此类推。
    
    # 筛选 NC (Normal Control)
    df_nc = df[df['NC'] == 1].copy()
    
    # 筛选 MCI (Mild Cognitive Impairment)
    df_mci = df[df['MCI'] == 1].copy()
    
    # 筛选 AD (Alzheimer's Disease)
    df_ad = df[df['AD'] == 1].copy()

    # 3. (可选) 如果你需要修改 path 列以匹配你旧数据的格式
    # 你的旧数据 path 是 '../../../MRI_data/AIBL_npy/'
    # 你的新数据 path 是 '/home/wcj/data/AIBL/'
    # 如果不需要修改路径，请注释掉下面这三行
    # target_path = '../../../MRI_data/AIBL_npy/'
    # df_nc['path'] = target_path
    # df_mci['path'] = target_path
    # df_ad['path'] = target_path

    # 4. 保存为三个新的 CSV 文件
    # index=False 表示保存时不保留 pandas 自动生成的行索引，保持格式整洁
    df_nc.to_csv('NC.csv', index=False)
    df_mci.to_csv('MCI.csv', index=False)
    df_ad.to_csv('AD.csv', index=False)

    # 5. 打印统计信息，确认切分结果
    print("-" * 30)
    print("切分完成！文件统计如下：")
    print(f"原始数据总行数: {len(df)}")
    print("-" * 30)
    print(f"NC.csv 保存成功，包含行数: {len(df_nc)}")
    print(f"MCI.csv 保存成功，包含行数: {len(df_mci)}")
    print(f"AD.csv  保存成功，包含行数: {len(df_ad)}")
    print("-" * 30)
    
    # 检查是否有重叠或遗漏（仅用于参考）
    total_split = len(df_nc) + len(df_mci) + len(df_ad)
    print(f"提取出的样本总数: {total_split}")
    if total_split < len(df):
        print(f"注意: 有 {len(df) - total_split} 行数据不属于 NC, MCI 或 AD (可能是其他标签如 OTHER, PD 等)。")

正在读取数据...
------------------------------
切分完成！文件统计如下：
原始数据总行数: 516
------------------------------
NC.csv 保存成功，包含行数: 358
MCI.csv 保存成功，包含行数: 86
AD.csv  保存成功，包含行数: 72
------------------------------
提取出的样本总数: 516


In [4]:
import os
import shutil
import pandas as pd
from tqdm import tqdm  # 用于显示进度条，如果没有安装可以通过 pip install tqdm 安装，或者删除相关代码

# ================= 配置区域 =================
# 源文件所在的目录
source_folder = 'MRI'

# 你的三个分类及其对应的CSV文件
# 键(Key)是文件夹名，值(Value)是CSV文件名
categories = {
    'NC': 'NC.csv',
    'MCI': 'MCI.csv',
    'AD': 'AD.csv'
}

# 操作模式: 'copy' (复制) 或 'move' (移动/剪切)
# 建议先用 'copy' 测试，确认无误后再改为 'move' 以节省空间
MODE = 'copy' 
# ===========================================

def organize_dataset():
    # 检查源目录是否存在
    if not os.path.exists(source_folder):
        print(f"错误: 源文件夹 '{source_folder}' 不存在，请检查路径。")
        return

    # 遍历三个分类
    for cat_name, csv_file in categories.items():
        print(f"\n正在处理分类: {cat_name} ...")
        
        # 1. 确保目标文件夹存在，如果不存在则创建
        if not os.path.exists(cat_name):
            os.makedirs(cat_name)
            print(f"  已创建文件夹: {cat_name}")
        
        # 2. 读取 CSV 文件
        if not os.path.exists(csv_file):
            print(f"  警告: 找不到 {csv_file}，跳过该分类。")
            continue
            
        try:
            df = pd.read_csv(csv_file)
        except Exception as e:
            print(f"  读取 CSV 失败: {e}")
            continue

        # 检查 filename 列是否存在
        if 'filename' not in df.columns:
            print(f"  错误: {csv_file} 中没有找到 'filename' 列。")
            continue

        # 3. 遍历 CSV 中的每一行文件名并移动/复制文件
        # 获取文件名列表
        file_list = df['filename'].tolist()
        
        success_count = 0
        missing_count = 0

        # 使用 tqdm 显示进度条 (如果你不想装 tqdm，可以将下行改为: for filename in file_list:)
        for filename in tqdm(file_list, desc=f"处理 {cat_name}", unit="file"):
            # 构建源文件路径和目标文件路径
            src_path = os.path.join(source_folder, filename)
            dst_path = os.path.join(cat_name, filename)
            
            # 检查源文件是否存在
            if os.path.exists(src_path):
                try:
                    # 执行复制或移动
                    if MODE == 'move':
                        shutil.move(src_path, dst_path)
                    else:
                        shutil.copy2(src_path, dst_path) # copy2 保留文件元数据
                    success_count += 1
                except Exception as e:
                    print(f"  操作失败 {filename}: {e}")
            else:
                # 如果源文件夹里找不到这个文件（可能是 CSV 里有但下载漏了，或者文件名不匹配）
                # 这种情况在目标文件夹里已经存在时也可能发生（如果是move模式且之前跑过）
                if os.path.exists(dst_path):
                     # 如果目标文件夹里已经有了，也算成功（防止重复运行报错）
                    success_count += 1
                else:
                    missing_count += 1
                    # print(f"  文件丢失: {filename}") # 如果丢失太多，可以取消注释查看具体文件名

        print(f"  -> 完成 {cat_name}: 成功 {success_count} 个, 缺失 {missing_count} 个")

if __name__ == "__main__":
    organize_dataset()
    print("\n所有操作完成！")


正在处理分类: NC ...


处理 NC: 100%|██████████████████████████████████████████████████████████████████| 358/358 [00:00<00:00, 13866.88file/s]


  -> 完成 NC: 成功 0 个, 缺失 358 个

正在处理分类: MCI ...


处理 MCI: 100%|████████████████████████████████████████████████████████████████████| 86/86 [00:00<00:00, 8468.96file/s]


  -> 完成 MCI: 成功 0 个, 缺失 86 个

正在处理分类: AD ...


处理 AD: 100%|███████████████████████████████████████████████████████████████████████████████| 72/72 [00:00<?, ?file/s]

  -> 完成 AD: 成功 0 个, 缺失 72 个

所有操作完成！
